# 00 — Setup: Build the SQLite Database

This notebook downloads the Olist dataset from Kaggle and loads all CSV files into a single SQLite database (`data/olist.db`).

**Run this once before any other notebook.**

## Prerequisites
1. Install the Kaggle CLI: `pip install kaggle`
2. Place your `kaggle.json` API token at `~/.kaggle/kaggle.json`
   - Get it from: https://www.kaggle.com/settings → API → Create New Token


In [ ]:
import subprocess, sqlite3, pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
DB_PATH  = DATA_DIR / 'olist.db'
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
# Download dataset from Kaggle
subprocess.run([
    'kaggle', 'datasets', 'download',
    '-d', 'olistbr/brazilian-ecommerce',
    '--unzip', '-p', str(DATA_DIR)
], check=True)
print('Download complete.')

In [ ]:
# Map CSV filenames to SQLite table names
TABLE_MAP = {
    'olist_orders_dataset.csv':                     'orders',
    'olist_order_items_dataset.csv':                'order_items',
    'olist_order_payments_dataset.csv':             'order_payments',
    'olist_order_reviews_dataset.csv':              'order_reviews',
    'olist_customers_dataset.csv':                  'customers',
    'olist_products_dataset.csv':                   'products',
    'olist_sellers_dataset.csv':                    'sellers',
    'olist_geolocation_dataset.csv':                'geolocation',
    'product_category_name_translation.csv':        'product_category_name_translation',
}

conn = sqlite3.connect(DB_PATH)

for filename, table_name in TABLE_MAP.items():
    filepath = DATA_DIR / filename
    if filepath.exists():
        df = pd.read_csv(filepath)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f'Loaded {table_name}: {len(df):,} rows')
    else:
        print(f'WARNING: {filename} not found, skipping.')

conn.close()
print(f'\nDatabase created at: {DB_PATH}')

In [ ]:
# Verify tables
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Tables in database:')
print(tables)
conn.close()